# Multi-Modal Medical Image Segmentation — Modality-Aware Routing Pipeline
**Projet : Segmentation sémantique d'images médicales — assistance au diagnostic radiologique**  
**Étudiant : Alaaeddine Bouchamla — Master 1 Génie Télécom, ENISO**  
**Encadrant : Pr. Mehrez Abdellaoui**  
**Partenaire clinique : Centre de Radiologie Émilie (CRE), Libreville, Gabon**

---

## Pipeline overview

A modality-aware routing system that dispatches each incoming volume to the
appropriate pretrained segmentation model, with shared downstream components
for anomaly detection and automated reporting.

### Modality coverage (based on CRE case mix, last 30 days)

| Clinical case        | Modality | Stage 1 model                                 |
|----------------------|----------|------------------------------------------------|
| TDM THORACIQUE       | CT       | `wholeBody_ct_segmentation` (104 organs)      |
| TDM TAP              | CT       | `wholeBody_ct_segmentation` (104 organs)      |
| TDM ABDO-PELVIEN     | CT       | `wholeBody_ct_segmentation` (104 organs)      |
| CT01–43              | CT       | `wholeBody_ct_segmentation` (104 organs)      |
| IRM CEREBRAL         | MRI T1   | `wholeBrainSeg_Large_UNEST_segmentation` (133)|
| IRM GENOU            | MRI      | MedSAM prompted segmentation (documented limitation) |
| IRM RACHIS CERVICAL  | MRI      | MedSAM prompted segmentation (documented limitation) |
| IRM EPAULE           | MRI      | MedSAM prompted segmentation (documented limitation) |
| IRM COUDE            | MRI      | MedSAM prompted segmentation (documented limitation) |

**Stage 1 (this notebook)**: load MONAI pretrained bundles for CT and brain MRI;
demonstrate inference. No training required.

**Stage 2 (separate)**: shared downstream components — anomaly-detection
autoencoder + French compte-rendu generation — implemented in the deployed
backend (`backend/ml/`).

### References

- Wasserthal et al., 2022. *TotalSegmentator: robust segmentation of 104 anatomical structures in CT images.* arXiv:2208.05868  
- Yu et al., 2023. *UNesT: local spatial representation learning with hierarchical transformer for efficient medical segmentation.* (Whole-brain MRI model)  
- Ma et al., 2024. *Segment Anything in Medical Images.* Nature Communications.


In [ ]:
# ── 1. Install dependencies ────────────────────────────────────────────────
# Toggle Internet ON in Kaggle's right sidebar (Session options) before running.
!pip install -q "monai[all]" "fire" nibabel tqdm scikit-learn
# torch, torchvision, matplotlib, numpy are pre-installed on Kaggle


In [ ]:
# ── 2. Imports + Google Drive (via gdown) ──────────────────────────────────
import os, random, time, shutil
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import jaccard_score
from tqdm import tqdm
from pathlib import Path

# ── Google Drive File IDs (right-click file in Drive → Share → Copy link) ────
GDRIVE_CKPT_ID    = ''   # unet_cre_finetuned checkpoint (leave blank for fresh)
GDRIVE_WEIGHTS_ID = ''   # unet_cre_finetuned best weights

# ── Paths ─────────────────────────────────────────────────────────────────────
WORKING_DIR  = '/kaggle/working'
BUNDLE_DIR   = f'{WORKING_DIR}/bundles'
SAVE_PATH    = f'{WORKING_DIR}/unet_cre_finetuned.pth'
CKPT_PATH    = f'{WORKING_DIR}/unet_cre_checkpoint.pth'

os.makedirs(BUNDLE_DIR, exist_ok=True)

if not os.path.exists(CKPT_PATH) and GDRIVE_CKPT_ID:
    import gdown
    print('Downloading checkpoint from Drive...')
    gdown.download(f'https://drive.google.com/uc?id={GDRIVE_CKPT_ID}', CKPT_PATH, quiet=False)

if not os.path.exists(SAVE_PATH) and GDRIVE_WEIGHTS_ID:
    import gdown
    print('Downloading best weights from Drive...')
    gdown.download(f'https://drive.google.com/uc?id={GDRIVE_WEIGHTS_ID}', SAVE_PATH, quiet=False)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Working dir : {WORKING_DIR}')
print(f'Bundle dir  : {BUNDLE_DIR}')


---
## Stage 1A — CT: Multi-Organ Pre-Segmentation (104 structures)

Downloads MONAI's `wholeBody_ct_segmentation` bundle. SegResNet-based,
trained on TotalSegmentator data (1,204 clinical CT scans). Two model
variants are bundled — we use the 3.0 mm low-resolution variant which
fits a Kaggle T4 GPU (the 1.5 mm variant requires ~27 GB VRAM).


In [ ]:
# ── 3. Download the MONAI wholeBody_ct_segmentation bundle ────────────────
# Bundle contains: model weights, config, inference pipeline, metadata.
# Two model variants: 1.5mm high-res (~27 GB GPU at inference) and 3.0mm
# low-res (fits T4). We use the low-res variant.

from monai.bundle import download

BUNDLE_NAME = 'wholeBody_ct_segmentation'

# Download (skip if already downloaded — checks for existing folder)
bundle_path = Path(BUNDLE_DIR) / BUNDLE_NAME
if not bundle_path.exists():
    print(f'Downloading {BUNDLE_NAME} bundle...')
    download(name=BUNDLE_NAME, bundle_dir=BUNDLE_DIR)
    print('Bundle downloaded.')
else:
    print(f'Bundle already at {bundle_path}')

# List bundle contents
for p in sorted(bundle_path.rglob('*')):
    if p.is_file():
        size_mb = p.stat().st_size / 1e6
        print(f'  {p.relative_to(bundle_path)}  ({size_mb:.1f} MB)')


In [ ]:
# ── 4. Stage 1A — Load CT model and run inference ──────────────────────────
# Uses the bundle's own config to instantiate the network and apply the
# correct preprocessing (HU windowing, spacing, cropping, etc.).

from monai.bundle import ConfigParser

bundle_path = Path(BUNDLE_DIR) / BUNDLE_NAME
config_path = bundle_path / 'configs' / 'inference.json'

parser = ConfigParser()
parser.read_config(str(config_path))
parser.read_meta(str(bundle_path / 'configs' / 'metadata.json'))

# Use 3.0 mm (low-res) variant — fits on free T4
parser['highres'] = False
parser['bundle_root'] = str(bundle_path)

# Instantiate the network and load weights
ct_network = parser.get_parsed_content('network').to(DEVICE)
ckpt = torch.load(str(bundle_path / 'models' / 'model_lowres.pt'),
                  map_location=DEVICE, weights_only=False)
state = ckpt.get('model', ckpt) if isinstance(ckpt, dict) else ckpt
ct_network.load_state_dict(state, strict=False)
ct_network.eval()

n_params = sum(p.numel() for p in ct_network.parameters())
print(f'Loaded {BUNDLE_NAME} (low-res, 3.0mm)')
print(f'Parameters  : {n_params:,} ({n_params/1e6:.1f}M)')
print(f'Output      : 105 channels (background + 104 anatomical structures)')
print(f'Architecture: SegResNet')
print(f'Modality    : CT')


---
## Stage 1B — Brain MRI: Whole-Brain Segmentation (133 structures)

Downloads MONAI's `wholeBrainSeg_Large_UNEST_segmentation` bundle. UNesT-based
(hierarchical transformer + 3D U-Net), trained to segment **133 brain
structures** from T1-weighted MRI. Covers IRM CEREBRAL cases.

**Architecture note**: UNesT is significantly heavier than SegResNet. On a free
T4 it works for inference but slowly (~30 s per volume). For deployment, the
backend caches results per-volume.


In [ ]:
# ── 5. Stage 1B — Download brain MRI bundle ────────────────────────────────
BRAIN_BUNDLE_NAME = 'wholeBrainSeg_Large_UNEST_segmentation'
brain_bundle_path = Path(BUNDLE_DIR) / BRAIN_BUNDLE_NAME

if not brain_bundle_path.exists():
    print(f'Downloading {BRAIN_BUNDLE_NAME} bundle...')
    download(name=BRAIN_BUNDLE_NAME, bundle_dir=BUNDLE_DIR)
    print('Bundle downloaded.')
else:
    print(f'Bundle already at {brain_bundle_path}')

# List contents
for p in sorted(brain_bundle_path.rglob('*')):
    if p.is_file():
        size_mb = p.stat().st_size / 1e6
        print(f'  {p.relative_to(brain_bundle_path)}  ({size_mb:.1f} MB)')


In [ ]:
# ── 6. Stage 1B — Load brain MRI model ─────────────────────────────────────
brain_config_path = brain_bundle_path / 'configs' / 'inference.json'

brain_parser = ConfigParser()
brain_parser.read_config(str(brain_config_path))
brain_parser.read_meta(str(brain_bundle_path / 'configs' / 'metadata.json'))
brain_parser['bundle_root'] = str(brain_bundle_path)

mri_network = brain_parser.get_parsed_content('network').to(DEVICE)

# Find the model checkpoint (filename varies between bundle versions)
model_files = list((brain_bundle_path / 'models').glob('*.pt'))
assert model_files, f'No .pt found in {brain_bundle_path}/models'
brain_ckpt = torch.load(str(model_files[0]), map_location=DEVICE, weights_only=False)
brain_state = brain_ckpt.get('model', brain_ckpt) if isinstance(brain_ckpt, dict) else brain_ckpt
mri_network.load_state_dict(brain_state, strict=False)
mri_network.eval()

n_params = sum(p.numel() for p in mri_network.parameters())
print(f'Loaded {BRAIN_BUNDLE_NAME}')
print(f'Parameters  : {n_params:,} ({n_params/1e6:.1f}M)')
print(f'Output      : 134 channels (background + 133 brain structures)')
print(f'Architecture: UNesT')
print(f'Modality    : MRI T1-weighted')


---
## Stage 1C — Modality-Aware Router

Routes each incoming volume to the right model based on a simple modality flag.
For CT cases without explicit modality metadata, we fall back to checking the
intensity range (CT volumes have HU-scale negative values; MRI does not).

For MSK MRI cases (knee, shoulder, elbow, spine), no public segmentation model
exists — these cases bypass Stage 1 and go directly to MedSAM prompted
segmentation in the deployed backend.


In [ ]:
# ── 7. Modality-aware routing ──────────────────────────────────────────────
import numpy as np
import torch

def detect_modality(volume: 'np.ndarray | torch.Tensor', metadata: dict | None = None) -> str:
    """Return one of: 'CT', 'MRI_BRAIN', 'MRI_MSK', 'UNKNOWN'.

    Prefer DICOM metadata when available; fall back to intensity heuristics.
    """
    if metadata is not None:
        modality = metadata.get('Modality', '').upper()
        body_part = metadata.get('BodyPartExamined', '').upper()
        if modality == 'CT':
            return 'CT'
        if modality in ('MR', 'MRI'):
            if any(k in body_part for k in ['BRAIN', 'HEAD', 'CEREBRAL', 'SKULL']):
                return 'MRI_BRAIN'
            if any(k in body_part for k in ['KNEE', 'SHOULDER', 'ELBOW', 'SPINE',
                                              'CERVICAL', 'GENOU', 'EPAULE',
                                              'COUDE', 'RACHIS']):
                return 'MRI_MSK'
            return 'MRI_BRAIN'  # default to brain for unspecified MRI

    # Intensity heuristic fallback: CT has negative HU values, MRI is non-negative
    arr = volume.cpu().numpy() if hasattr(volume, 'cpu') else np.asarray(volume)
    if arr.min() < -200:
        return 'CT'
    return 'UNKNOWN'

def route_and_segment(volume, metadata=None):
    """Dispatch a volume to the right Stage 1 model. Returns (modality, output).

    For MSK MRI, returns (modality, None) — handled by MedSAM downstream.
    """
    modality = detect_modality(volume, metadata)
    if modality == 'CT':
        return modality, ct_network        # Stage 1A
    elif modality == 'MRI_BRAIN':
        return modality, mri_network       # Stage 1B
    elif modality == 'MRI_MSK':
        return modality, None              # Defer to MedSAM
    else:
        return modality, None

print('Modality router ready.')
print('Available routes:')
print('  CT          → wholeBody_ct_segmentation  (104 organs)')
print('  MRI_BRAIN   → wholeBrainSeg              (133 structures)')
print('  MRI_MSK     → MedSAM prompted (deferred to backend)')
print('  UNKNOWN     → flagged for radiologist review')


In [ ]:
# ── 8. Stage 1 ready ───────────────────────────────────────────────────────
print('Stage 1 components loaded and ready for inference.')
print()
print('Bundles in /kaggle/working/bundles/:')
for d in sorted(Path(BUNDLE_DIR).iterdir()):
    if d.is_dir():
        total = sum(f.stat().st_size for f in d.rglob('*') if f.is_file())
        print(f'  {d.name:<50} {total/1e6:.1f} MB')

print()
print('Next steps (handled in the backend, not this notebook):')
print('  - Modality detection from DICOM metadata')
print('  - Per-organ volumetric measurements (mL, mean HU/intensity)')
print('  - Anomaly detection via convolutional autoencoder')
print('  - Automated French compte-rendu generation')
print('  - MedSAM prompted segmentation for MSK MRI cases')


---
## Stage 2 — CRE Fine-Tuning (2D U-Net on MedSAM Pseudo-Labels)

A lightweight 2D U-Net fine-tuned on real CRE patient DICOM slices using
MedSAM-generated pseudo-labels. This is a documented transfer-learning
experiment: honest expectation is modest improvement over random init, not
better than Stage 1 pretrained models.

**Architecture**: 2D U-Net — in_channels=1, num_classes=2 (binary), features=[64,128,256,512]  
**Loss**: Combined Dice + Cross-Entropy (α=0.5)  
**Optimizer**: AdamW, lr=5e-5, weight_decay=1e-5  
**Scheduler**: CosineAnnealingLR, T_max=25  
**Epochs**: 25 | **Checkpoint every**: 3 epochs | **Early stopping patience**: 6

**Data**: MedSAM pseudo-labels generated by `backend/ml/medsam/pseudo_labeler.py`.
Upload `pseudo_labels.zip` as a Kaggle Dataset and set `PSEUDO_DATASET_SLUG` below,
or upload manually and extract to `/kaggle/working/pseudo_labels/`.


In [ ]:
# ── Stage 2 · Step 1: U-Net architecture + Loss + Metrics ──────────────────
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np
from sklearn.metrics import jaccard_score, precision_score, recall_score

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch), nn.ReLU(inplace=True),
        )
    def forward(self, x): return self.net(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(nn.MaxPool2d(2), DoubleConv(in_ch, out_ch))
    def forward(self, x): return self.net(x)

class Up(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.up   = nn.ConvTranspose2d(in_ch, in_ch // 2, 2, stride=2)
        self.conv = DoubleConv(in_ch, out_ch)
    def forward(self, x1, x2):
        x1 = self.up(x1)
        dh = x2.size(2) - x1.size(2)
        dw = x2.size(3) - x1.size(3)
        x1 = F.pad(x1, [dw // 2, dw - dw // 2, dh // 2, dh - dh // 2])
        return self.conv(torch.cat([x2, x1], dim=1))

class UNet(nn.Module):
    def __init__(self, in_channels=1, num_classes=2, features=[64, 128, 256, 512]):
        super().__init__()
        self.inc      = DoubleConv(in_channels, features[0])
        self.downs    = nn.ModuleList([Down(features[i], features[i+1]) for i in range(len(features)-1)])
        self.bottleneck = DoubleConv(features[-1], features[-1]*2)
        self.ups      = nn.ModuleList()
        ch = features[-1] * 2
        for f in reversed(features):
            self.ups.append(Up(ch, f))
            ch = f
        self.outc = nn.Conv2d(features[0], num_classes, 1)

    def forward(self, x):
        skips = [self.inc(x)]
        for d in self.downs:
            skips.append(d(skips[-1]))
        x = self.bottleneck(skips[-1])
        for up, skip in zip(self.ups, reversed(skips)):
            x = up(x, skip)
        return self.outc(x)


class DiceLoss(nn.Module):
    def __init__(self, smooth=1e-5):
        super().__init__()
        self.smooth = smooth
    def forward(self, logits, targets):
        probs      = torch.softmax(logits, dim=1)
        targets_oh = F.one_hot(targets, num_classes=logits.shape[1]).permute(0,3,1,2).float()
        dims       = (0, 2, 3)
        inter      = (probs * targets_oh).sum(dims)
        card       = (probs + targets_oh).sum(dims)
        return 1 - ((2. * inter + self.smooth) / (card + self.smooth)).mean()

class CombinedLoss(nn.Module):
    def __init__(self, alpha=0.5):
        super().__init__()
        self.dice = DiceLoss()
        self.ce   = nn.CrossEntropyLoss()
        self.alpha = alpha
    def forward(self, logits, targets):
        return self.alpha * self.dice(logits, targets) + (1 - self.alpha) * self.ce(logits, targets)


def compute_metrics(pred, target, num_classes=2):
    p = pred.cpu().numpy().flatten()
    t = target.cpu().numpy().flatten()
    iou  = jaccard_score(t, p, average='macro', zero_division=0)
    dice_scores = []
    for c in range(num_classes):
        pc, tc = (p == c), (t == c)
        inter  = (pc & tc).sum()
        denom  = pc.sum() + tc.sum()
        dice_scores.append((2 * inter / denom) if denom > 0 else 1.0)
    prec = precision_score(t, p, average='macro', zero_division=0)
    rec  = recall_score(t, p, average='macro', zero_division=0)
    return {'iou': iou, 'dice': float(np.mean(dice_scores)), 'precision': prec, 'recall': rec}

print('U-Net, losses, and metrics defined.')


In [ ]:
# ── Stage 2 · Step 2: CRE Dataset loader ───────────────────────────────────
import json, random, cv2
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

# ── Where are the pseudo-labels? ─────────────────────────────────────────────
# Option A: added as a Kaggle Dataset → set the dataset slug (e.g. 'youruser/pseudo-labels-cre')
PSEUDO_DATASET_SLUG = ''   # set this, then click Add Data → Your Datasets in Kaggle

# Option B: uploaded manually via Files panel → /kaggle/working/pseudo_labels/
PSEUDO_LABEL_ROOT = (
    Path(f'/kaggle/input/{PSEUDO_DATASET_SLUG}')
    if PSEUDO_DATASET_SLUG
    else Path('/kaggle/working/pseudo_labels')
)

assert PSEUDO_LABEL_ROOT.exists(), (
    f'Pseudo-label root not found: {PSEUDO_LABEL_ROOT}\n'
    'Either set PSEUDO_DATASET_SLUG or upload pseudo_labels/ manually.'
)

all_entries = []
for manifest_file in PSEUDO_LABEL_ROOT.rglob('manifest.json'):
    with open(manifest_file) as f:
        all_entries.extend(json.load(f))

print(f'Pseudo-label pairs found: {len(all_entries)}')
assert len(all_entries) > 0, 'No entries found in manifest.json files'


class CREDataset(Dataset):
    TARGET_SIZE = 512

    def __init__(self, entries, augment=False):
        self.entries = entries
        self.augment = augment

    def __len__(self):
        return len(self.entries)

    def __getitem__(self, idx):
        e   = self.entries[idx]
        img = np.load(e['slice']).astype(np.float32)
        msk = np.load(e['mask']).astype(np.int64)

        if img.shape != (self.TARGET_SIZE, self.TARGET_SIZE):
            img = cv2.resize(img, (self.TARGET_SIZE, self.TARGET_SIZE), interpolation=cv2.INTER_LINEAR)
            msk = cv2.resize(msk.astype(np.uint8), (self.TARGET_SIZE, self.TARGET_SIZE),
                             interpolation=cv2.INTER_NEAREST).astype(np.int64)

        img_t = torch.from_numpy(img).unsqueeze(0)  # (1, H, W)
        msk_t = torch.from_numpy(msk)               # (H, W)

        if self.augment and random.random() > 0.5:
            img_t = torch.flip(img_t, dims=[2]); msk_t = torch.flip(msk_t, dims=[1])
        if self.augment and random.random() > 0.5:
            img_t = torch.flip(img_t, dims=[1]); msk_t = torch.flip(msk_t, dims=[0])

        return img_t, msk_t


random.shuffle(all_entries)
n_train = int(0.8 * len(all_entries))
train_entries, val_entries = all_entries[:n_train], all_entries[n_train:]

cre_train_loader = DataLoader(CREDataset(train_entries, augment=True),
                               batch_size=4, shuffle=True,  num_workers=2)
cre_val_loader   = DataLoader(CREDataset(val_entries,   augment=False),
                               batch_size=2, shuffle=False, num_workers=2)

print(f'Train: {len(train_entries)} slices | Val: {len(val_entries)} slices')
print(f'Train batches: {len(cre_train_loader)} | Val batches: {len(cre_val_loader)}')


In [ ]:
# ── Stage 2 · Step 3: Training loop with checkpoint resume ─────────────────
import time

NUM_EPOCHS  = 25
SAVE_EVERY  = 3
PATIENCE    = 6
LR          = 5e-5

cre_model   = UNet(in_channels=1, num_classes=2).to(DEVICE)
criterion   = CombinedLoss(alpha=0.5)
optimizer   = torch.optim.AdamW(cre_model.parameters(), lr=LR, weight_decay=1e-5)
scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

best_dice   = 0.0
no_improve  = 0
history     = {'loss': [], 'val_loss': [], 'iou': [], 'dice': [], 'precision': [], 'recall': []}
start_epoch = 1

# Resume from checkpoint if available (downloaded from Drive in Cell 2)
if Path(CKPT_PATH).exists():
    print('Checkpoint found — resuming...')
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE, weights_only=False)
    cre_model.load_state_dict(ckpt['model'])
    optimizer.load_state_dict(ckpt['optimizer'])
    scheduler.load_state_dict(ckpt['scheduler'])
    best_dice   = float(ckpt['best_dice'])
    no_improve  = int(ckpt.get('no_improve', 0))
    history     = ckpt['history']
    start_epoch = int(ckpt['epoch']) + 1
    print(f'  Resumed from epoch {ckpt["epoch"]} | Best Dice: {best_dice:.4f}')
else:
    print('No checkpoint — starting from epoch 1.')


def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    total = 0
    for imgs, masks in tqdm(loader, desc='Train', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        logits = model(imgs)
        loss   = criterion(logits, masks)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)

@torch.no_grad()
def validate(model, loader):
    model.eval()
    total_loss = 0
    ious, dices, precs, recs = [], [], [], []
    for imgs, masks in tqdm(loader, desc='Val', leave=False):
        imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
        logits = model(imgs)
        total_loss += criterion(logits, masks).item()
        m = compute_metrics(logits.argmax(dim=1), masks)
        ious.append(m['iou']); dices.append(m['dice'])
        precs.append(m['precision']); recs.append(m['recall'])
    return (total_loss / len(loader),
            float(np.mean(ious)), float(np.mean(dices)),
            float(np.mean(precs)), float(np.mean(recs)))


print(f'Training epochs {start_epoch}–{NUM_EPOCHS} on {DEVICE}...\n')

for epoch in range(start_epoch, NUM_EPOCHS + 1):
    t0 = time.time()
    train_loss = train_one_epoch(cre_model, cre_train_loader, criterion, optimizer)
    val_loss, iou, dice, prec, rec = validate(cre_model, cre_val_loader)
    scheduler.step()

    history['loss'].append(float(train_loss))
    history['val_loss'].append(float(val_loss))
    history['iou'].append(float(iou))
    history['dice'].append(float(dice))
    history['precision'].append(float(prec))
    history['recall'].append(float(rec))

    print(f'Epoch {epoch:2d}/{NUM_EPOCHS} | '
          f'Loss {train_loss:.4f} | Val {val_loss:.4f} | '
          f'IoU {iou:.4f} | Dice {dice:.4f} | '
          f'Prec {prec:.4f} | Rec {rec:.4f} | {time.time()-t0:.0f}s')

    if dice > best_dice:
        best_dice  = float(dice)
        no_improve = 0
        torch.save(cre_model.state_dict(), SAVE_PATH)
        print(f'  -> Best weights saved (Dice={best_dice:.4f})')
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'  Early stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
            break

    if epoch % SAVE_EVERY == 0:
        torch.save({
            'epoch': int(epoch),
            'model': cre_model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_dice': float(best_dice),
            'no_improve': int(no_improve),
            'history': history,
        }, CKPT_PATH)
        print(f'  -> Checkpoint saved (epoch {epoch})')

print(f'\nDone. Best Dice: {best_dice:.4f}')
print(f'Weights: {SAVE_PATH}')
print(f'Checkpoint: {CKPT_PATH}')
print('\n>>> Download both files from the Output tab and upload to Google Drive.')
print('>>> Paste the Drive file IDs into GDRIVE_CKPT_ID / GDRIVE_WEIGHTS_ID in Cell 2 to resume next session.')


In [ ]:
# ── Stage 2 · Step 4: Plot curves + print final metrics ────────────────────
import matplotlib.pyplot as plt

epochs_range = range(1, len(history['loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(epochs_range, history['loss'],     label='Train', color='#1a3c6e')
axes[0].plot(epochs_range, history['val_loss'], label='Val',   color='#c0392b')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['iou'],  label='IoU',  color='#1a3c6e')
axes[1].plot(epochs_range, history['dice'], label='Dice', color='#c0392b')
axes[1].set_title('IoU & Dice (val)'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_range, history['precision'], label='Precision', color='#1a3c6e')
axes[2].plot(epochs_range, history['recall'],    label='Recall',    color='#c0392b')
axes[2].set_title('Precision & Recall (val)'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.suptitle(f'Stage 2 Fine-tuning — CRE Pseudo-Labels | Best Dice={best_dice:.4f}', fontsize=12)
plt.tight_layout()
plot_path = f'{WORKING_DIR}/stage2_curves.png'
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()

print('\n=== STAGE 2 FINAL RESULTS (copy into article Table I) ===')
print(f'  IoU       : {history["iou"][-1]:.4f}')
print(f'  Dice      : {history["dice"][-1]:.4f}  (best={best_dice:.4f})')
print(f'  Precision : {history["precision"][-1]:.4f}')
print(f'  Recall    : {history["recall"][-1]:.4f}')
print(f'\nFiles ready to download from Output tab:')
print(f'  {SAVE_PATH}')
print(f'  {CKPT_PATH}')
print(f'  {plot_path}')
print('\nPlace unet_cre_finetuned.pth in the project models/ directory.')
